In [2]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import normflows as nf

import time
from torch.distributions import Normal

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class NBase(nn.Module):
    def __init__(self, dim, init_loc=0.0, init_log_scale=-2.5):
        super().__init__()
        self.dim = dim
        self.loc = nn.Parameter(torch.full((dim,), float(init_loc)))
        self.raw_log_scale = nn.Parameter(torch.full((dim,), float(init_log_scale)))

    def log_scale(self):
        return torch.clamp(self.raw_log_scale, min=-5.0, max=2.0)

    def scale(self):
        return torch.exp(self.log_scale())

    def rsample(self, num_samples):
        eta = torch.randn(
            num_samples,
            self.dim,
            device=self.loc.device,
            dtype=self.loc.dtype,
        )
        z0 = self.loc.unsqueeze(0) + self.scale().unsqueeze(0) * eta
        return eta, z0

    def log_prob(self, z0):
        log_scale = self.log_scale().unsqueeze(0)
        var = torch.exp(2.0 * log_scale)
        return -0.5 * (
            ((z0 - self.loc.unsqueeze(0)) ** 2) / var
            + 2.0 * log_scale
            + math.log(2.0 * math.pi)
        ).sum(dim=1)

In [4]:
def simfun1(n=180, p=100, seed=123, snr=3.0, true_prop=0.1, device=None, dtype=torch.float32,):

    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)

    X = rng.standard_normal((n, p)).astype(np.float32)
    X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-8)

    n_active = int(p * true_prop)
    active_idx = np.sort(rng.choice(p, size=n_active, replace=False))

    beta_true = np.zeros(p, dtype=np.float32)
    magnitudes = rng.uniform(0.3, 2.0, size=n_active).astype(np.float32)
    signs = rng.choice([-1.0, 1.0], size=n_active).astype(np.float32)
    beta_true[active_idx] = signs * magnitudes

    signal = X @ beta_true
    sigma2 = np.var(signal) / snr
    sigma = np.sqrt(sigma2)

    y = signal + sigma * rng.standard_normal(n).astype(np.float32)
    y = y - y.mean()

    X_t = torch.tensor(X, dtype=dtype, device=device)
    y_t = torch.tensor(y, dtype=dtype, device=device)
    beta_true_t = torch.tensor(beta_true, dtype=dtype, device=device)

    info = {"n": n, "p": p, "n_active": n_active, "sigma2": float(sigma2), "sigma": float(sigma), "active_idx": active_idx, "snr": snr,}

    return X_t, y_t, beta_true_t, info

In [14]:
class SemanticAffineCoupling(nn.Module):
    """
    Affine coupling layer on semantic blocks of xi = [s, u, t].

    Mode:
        - "s": update s conditioned on (u, t)
        - "u": update u conditioned on (s, t)
        - "t": update t conditioned on (s, u)

    xi shape: [batch, 2p + 1]
    ordering : [ s(0:p) | u(p:2p) | t(2p) ]
    """
    def __init__(self, p, mode, hidden_units=128, num_hidden_layers=2, scale_clip=2.0):
        super().__init__()
        assert mode in {"s", "u", "t"}
        self.p = int(p)
        self.dim = 2 * p + 1
        self.mode = mode
        self.scale_clip = float(scale_clip)

        if mode == "s":
            cond_dim = p + 1      # (u, t)
            trans_dim = p         # s
        elif mode == "u":
            cond_dim = p + 1      # (s, t)
            trans_dim = p         # u
        else:  # mode == "t"
            cond_dim = 2 * p      # (s, u)
            trans_dim = 1         # t

        widths = [cond_dim] + [hidden_units] * num_hidden_layers + [2 * trans_dim]
        self.net = nf.nets.MLP(widths, init_zeros=True)

    def _split(self, x):
        s = x[:, :self.p]
        u = x[:, self.p:2 * self.p]
        t = x[:, 2 * self.p:2 * self.p + 1]
        return s, u, t

    def _merge(self, s, u, t):
        return torch.cat([s, u, t], dim=-1)

    def _affine_params(self, cond):
        h = self.net(cond)
        log_scale, shift = torch.chunk(h, 2, dim=-1)
        log_scale = self.scale_clip * torch.tanh(log_scale / self.scale_clip)
        return log_scale, shift

    def forward(self, x, return_logdet=False):
        s, u, t = self._split(x)

        if self.mode == "s":
            cond = torch.cat([u, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            s_new = s * torch.exp(log_scale) + shift
            y = self._merge(s_new, u, t)
            logdet = log_scale.sum(dim=-1)

        elif self.mode == "u":
            cond = torch.cat([s, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            u_new = u * torch.exp(log_scale) + shift
            y = self._merge(s, u_new, t)
            logdet = log_scale.sum(dim=-1)

        else:  # mode == "t"
            cond = torch.cat([s, u], dim=-1)
            log_scale, shift = self._affine_params(cond)
            t_new = t * torch.exp(log_scale) + shift
            y = self._merge(s, u, t_new)
            logdet = log_scale.sum(dim=-1)

        if return_logdet:
            return y, logdet
        return y

    def inverse(self, y, return_logdet=False):
        s, u, t = self._split(y)

        if self.mode == "s":
            cond = torch.cat([u, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            s_old = (s - shift) * torch.exp(-log_scale)
            x = self._merge(s_old, u, t)
            logdet = (-log_scale).sum(dim=-1)

        elif self.mode == "u":
            cond = torch.cat([s, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            u_old = (u - shift) * torch.exp(-log_scale)
            x = self._merge(s, u_old, t)
            logdet = (-log_scale).sum(dim=-1)

        else:  # mode == "t"
            cond = torch.cat([s, u], dim=-1)
            log_scale, shift = self._affine_params(cond)
            t_old = (t - shift) * torch.exp(-log_scale)
            x = self._merge(s, u, t_old)
            logdet = (-log_scale).sum(dim=-1)

        if return_logdet:
            return x, logdet
        return x


class FlowMap(nn.Module):
    def __init__(self, p=None, dim=None, K=4,
                 hidden_units=128, num_hidden_layers=2, scale_clip=2.0):
        super().__init__()

        if p is None:
            if dim is None:
                raise ValueError("Either p or dim must be provided.")
            if (dim - 1) % 2 != 0:
                raise ValueError("dim must equal 2*p + 1.")
            p = (dim - 1) // 2

        self.p = int(p)
        self.dim = 2 * self.p + 1

        layers = []
        for _ in range(K):
            layers.append(
                SemanticAffineCoupling(
                    p=self.p, mode="s",
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                )
            )
            layers.append(
                SemanticAffineCoupling(
                    p=self.p, mode="u",
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                )
            )
            layers.append(
                SemanticAffineCoupling(
                    p=self.p, mode="t",
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                )
            )

        self.layers = nn.ModuleList(layers)

    def forward(self, x, return_logdet=False):
        z = x
        if return_logdet:
            total_logdet = x.new_zeros(x.shape[0])

        for layer in self.layers:
            if return_logdet:
                z, logdet = layer(z, return_logdet=True)
                total_logdet = total_logdet + logdet
            else:
                z = layer(z)

        if return_logdet:
            return z, total_logdet
        return z

    def inverse(self, z, return_logdet=False):
        x = z
        if return_logdet:
            total_logdet = z.new_zeros(z.shape[0])

        for layer in reversed(self.layers):
            if return_logdet:
                x, logdet = layer.inverse(x, return_logdet=True)
                total_logdet = total_logdet + logdet
            else:
                x = layer.inverse(x)

        if return_logdet:
            return x, total_logdet
        return x

In [6]:
class Relaxedsas(nn.Module):
    def __init__(self, X, y, sigma2, tau, g_theta):
        super().__init__()
        if g_theta is None:
            raise ValueError("g_theta must be provided.")

        self.register_buffer("X", X)
        self.register_buffer("y", y)
        self.register_buffer("sigma2", torch.tensor(float(sigma2), dtype=X.dtype, device=X.device))
        self.register_buffer("tau", torch.tensor(float(tau), dtype=X.dtype, device=X.device))

        self.n, self.p = X.shape
        self.dim = 2 * self.p + 1
        self.g_theta = g_theta

    def set_tau(self, tau):
        self.tau.fill_(float(tau))

    def decode(self, eps):
        xi = self.g_theta(eps)

        p = self.p
        s = xi[:, :p]
        u = xi[:, p:2 * p]
        t = xi[:, 2 * p:2 * p + 1]

        gate = torch.sigmoid((u - t) / self.tau)
        beta = s * gate

        return {"eps": eps, "xi": xi, "s": s, "u": u, "t": t, "gate": gate, "beta": beta,}

    def log_joint(self, eps):
        dec = self.decode(eps)
        beta = dec["beta"]

        fit = self.X @ beta.T
        resid = self.y[:, None] - fit

        loglik = -0.5 * (
            resid.pow(2).sum(dim=0) / self.sigma2
            + self.n * torch.log(2.0 * torch.pi * self.sigma2)
        )

        log_p0_eps = -0.5 * (
            eps.pow(2) + math.log(2.0 * math.pi)
        ).sum(dim=1)

        return loglik + log_p0_eps

In [12]:
class FlowVI(nn.Module):
    def __init__(self, q0, posterior_flow, generative_model):
        super().__init__()
        self.q0 = q0
        self.posterior_flow = posterior_flow
        self.generative_model = generative_model

    def sample_posterior(self, num_samples):
        _, z0 = self.q0.rsample(num_samples)
        eps, logdet = self.posterior_flow(z0, return_logdet=True)
        log_q_eps = self.q0.log_prob(z0) - logdet
        return eps, log_q_eps

    def neg_elbo(self, num_samples=256, elbo_beta=1.0):
        eps, log_q_eps = self.sample_posterior(num_samples)
        log_joint = self.generative_model.log_joint(eps)
        return (log_q_eps - elbo_beta * log_joint).mean()


def build_flow_vi(X, y, sigma2, tau, K_q=8, K_g=8, hidden_units=64, num_hidden_layers=2,):
    p = X.shape[1]
    dim = 2 * p + 1

    g_theta = FlowMap(p=X.shape[1], K=K_g, hidden_units=hidden_units,
                          num_hidden_layers=num_hidden_layers, scale_clip=2.0)

    generative_model = Relaxedsas(X=X, y=y, sigma2=sigma2, tau=tau, g_theta=g_theta,)

    q0 = NBase(dim=dim, init_loc=0.0, init_log_scale=-2.5,)
    
    posterior_flow = FlowMap(
        dim=dim,
        K=K_q,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
        scale_clip=2.0,
    )

    return FlowVI(q0=q0, posterior_flow=posterior_flow, generative_model=generative_model,)

In [8]:
def train_flow(model, epochs=1000, num_samples=128, lr=5e-5, tau_start=1.0, tau_end=0.05,
    anneal_ratio=0.7, grad_clip=3.0, print_every=100, elbo_beta=1.0,):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    tau_hist = []
    grad_hist = []
    
    anneal_epochs = max(1, int(round(epochs * anneal_ratio)))

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        if epoch <= anneal_epochs:
            frac = (epoch - 1) / max(anneal_epochs - 1, 1)
            tau_now = tau_start * (tau_end / tau_start) ** frac
        else:
            tau_now = float(tau_end)

        model.generative_model.set_tau(tau_now)
        tau_hist.append(float(tau_now))

        loss = model.neg_elbo(num_samples=num_samples, elbo_beta=elbo_beta)
        loss.backward()

        if grad_clip is not None:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            grad_norm = float(grad_norm)
        else:
            grad_norm = 0.0

        optimizer.step()

        losses.append(float(loss.item()))
        grad_hist.append(grad_norm)
        
        with torch.no_grad():
            point_trace = trace_eps_point(model)
            u = point_trace["u_hat"].squeeze(0)
            t = point_trace["t_hat"].squeeze(0)
            marg = u - t
            n_selected = int((marg > 0).sum().item())
            near_boundary = int((marg.abs() < 0.05).sum().item())

        
        if epoch % print_every == 0 or epoch == 1:
            print(
                f"[epoch {epoch:04d}] "
                f"loss={loss.item():.6f} "
                f"tau={tau_now:.6f} "
                f"grad_norm={grad_norm:.6f} "
                f"n_selected={n_selected} "
                f"near_boundary={near_boundary}"
            )

    return losses, tau_hist, grad_hist

In [9]:
@torch.no_grad()
def trace_eps_point(model):
    z0_hat = model.q0.loc.unsqueeze(0)
    eps_hat, _ = model.posterior_flow(z0_hat, return_logdet=True)
    dec = model.generative_model.decode(eps_hat)

    return {
        "s_hat": dec["s"].detach().cpu(),
        "u_hat": dec["u"].detach().cpu(),
        "t_hat": dec["t"].detach().cpu(),
        "beta_hat": dec["beta"].detach().cpu(),
    }

@torch.no_grad()
def point_selection_summary(point_trace, beta_true):
    u_hat = point_trace["u_hat"].squeeze(0)
    t_hat = point_trace["t_hat"].squeeze(0)
    beta_hat = point_trace["beta_hat"].squeeze(0)

    beta_true_cpu = beta_true.detach().cpu()

    pred = (u_hat > t_hat).int().numpy()
    truth = (beta_true_cpu.abs() > 1e-12).int().numpy()

    tp = int(((pred == 1) & (truth == 1)).sum())
    fp = int(((pred == 1) & (truth == 0)).sum())
    fn = int(((pred == 0) & (truth == 1)).sum())
    tn = int(((pred == 0) & (truth == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    if t_hat.numel() == 1:
        t_col = np.repeat(t_hat.item(), u_hat.numel())
    else:
        t_col = t_hat.numpy()

    df = pd.DataFrame({"j": np.arange(u_hat.numel()), "beta_true": beta_true_cpu.numpy(),
        "u_hat": u_hat.numpy(), "t_hat": t_col, "beta_hat": beta_hat.numpy(),
        "truth": truth, "pred": pred,
    })

    metrics = {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision, "recall": recall, "f1": f1,
        "n_selected": int(pred.sum()), "selected_idx": np.where(pred == 1)[0].tolist(),
    }

    return df, metrics

In [16]:
def simflow(seed=123, device=None, dtype=torch.float32, hidden_units=64, num_hidden_layers=2,
    n=180, p=100, snr=3.0, true_prop=0.1, tau_end=0.5, K_q=8, K_g=8, anneal_ratio=0.8,
    epochs=1000, num_samples=128, lr=5e-5, grad_clip=3.0, print_every=100,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X, y, beta_true, sim_info = simfun1(n=n, p=p, seed=seed, snr=snr, true_prop=true_prop, device=device, dtype=dtype,)

    print("===== Simulation info =====")
    print(sim_info)

    model = build_flow_vi(X=X, y=y, sigma2=sim_info["sigma2"], tau=tau_end, K_q=K_q, K_g=K_g,
        hidden_units=hidden_units, num_hidden_layers=num_hidden_layers,
    ).to(device)

    losses, tau_hist, grad_hist = train_flow(model=model, epochs=epochs, num_samples=num_samples,
        lr=lr, tau_start=1.0, tau_end=tau_end, anneal_ratio=anneal_ratio, grad_clip=grad_clip, print_every=print_every, elbo_beta=1.0,
    )

    point_trace = trace_eps_point(model)
    selection_df, metrics = point_selection_summary(point_trace, beta_true)

    return {"beta_true": beta_true, "sim_info": sim_info, "model": model, "losses": losses,
        "tau_hist": tau_hist, "grad_hist": grad_hist, "point_trace": point_trace,
        "selection_df": selection_df, "metrics": metrics,
    }

In [18]:
out_flow = simflow(seed=123, n=120, p=100, snr=2.5, true_prop=0.1, anneal_ratio=0.75,
    tau_end=0.01, K_q=16, K_g=16, hidden_units=64, num_hidden_layers=2,
    epochs=10000, num_samples=128, lr=5e-5, grad_clip=3.0, print_every=500,
)

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 6.834272766113282, 'sigma': 2.614244205523516, 'active_idx': array([14, 23, 24, 28, 29, 45, 69, 74, 84, 90], dtype=int64), 'snr': 2.5}
[epoch 0001] loss=894.577026 tau=1.000000 grad_norm=386.875793 n_selected=0 near_boundary=100
[epoch 0500] loss=249.406555 tau=0.736064 grad_norm=130.564865 n_selected=14 near_boundary=1
[epoch 1000] loss=244.576340 tau=0.541458 grad_norm=83.000130 n_selected=7 near_boundary=4
[epoch 1500] loss=242.985016 tau=0.398303 grad_norm=135.879837 n_selected=14 near_boundary=5
[epoch 2000] loss=242.845032 tau=0.292996 grad_norm=84.639954 n_selected=19 near_boundary=3
[epoch 2500] loss=242.525116 tau=0.215532 grad_norm=160.912613 n_selected=19 near_boundary=4
[epoch 3000] loss=242.592087 tau=0.158548 grad_norm=126.101608 n_selected=20 near_boundary=5
[epoch 3500] loss=242.454987 tau=0.116630 grad_norm=204.242752 n_selected=23 near_boundary=9
[epoch 4000] loss=242.401260 tau=0.085794 grad_n

In [19]:
class HierarchicalSemanticCoupling(nn.Module):
    """
    Two-level semantic affine coupling for xi = [s | u | t].

    Modes
    -----
    "s_odd"  : update s_odd  given (s_even, u, t)
    "s_even" : update s_even given (s_odd,  u, t)
    "u_odd"  : update u_odd  given (u_even, s, t)
    "u_even" : update u_even given (u_odd,  s, t)
    "t"      : update t given (s, u)

    Ordering of xi
    --------------
    xi = [ s(0:p) | u(p:2p) | t(2p) ]
    """

    def __init__(
        self,
        p,
        mode,
        hidden_units=128,
        num_hidden_layers=2,
        scale_clip=2.0,
        t_update="additive",   # "additive" or "affine"
    ):
        super().__init__()
        assert mode in {"s_odd", "s_even", "u_odd", "u_even", "t"}
        assert t_update in {"additive", "affine"}

        self.p = int(p)
        self.dim = 2 * self.p + 1
        self.mode = mode
        self.scale_clip = float(scale_clip)
        self.t_update = t_update

        even_idx = torch.arange(0, self.p, 2, dtype=torch.long)
        odd_idx = torch.arange(1, self.p, 2, dtype=torch.long)
        self.register_buffer("even_idx", even_idx)
        self.register_buffer("odd_idx", odd_idx)

        n_even = len(even_idx)
        n_odd = len(odd_idx)

        if mode == "s_odd":
            cond_dim = n_even + self.p + 1      # s_even, u, t
            trans_dim = n_odd
            out_dim = 2 * trans_dim

        elif mode == "s_even":
            cond_dim = n_odd + self.p + 1       # s_odd, u, t
            trans_dim = n_even
            out_dim = 2 * trans_dim

        elif mode == "u_odd":
            cond_dim = n_even + self.p + 1      # u_even, s, t
            trans_dim = n_odd
            out_dim = 2 * trans_dim

        elif mode == "u_even":
            cond_dim = n_odd + self.p + 1       # u_odd, s, t
            trans_dim = n_even
            out_dim = 2 * trans_dim

        else:  # mode == "t"
            cond_dim = 2 * self.p               # s, u
            trans_dim = 1
            out_dim = 1 if t_update == "additive" else 2

        widths = [cond_dim] + [hidden_units] * num_hidden_layers + [out_dim]
        self.net = nf.nets.MLP(widths, init_zeros=True)

    def _split(self, x):
        s = x[:, :self.p]
        u = x[:, self.p:2 * self.p]
        t = x[:, 2 * self.p:2 * self.p + 1]
        return s, u, t

    def _merge(self, s, u, t):
        return torch.cat([s, u, t], dim=-1)

    def _affine_params(self, cond):
        h = self.net(cond)
        log_scale, shift = torch.chunk(h, 2, dim=-1)
        log_scale = self.scale_clip * torch.tanh(log_scale / self.scale_clip)
        return log_scale, shift

    def _t_params(self, cond):
        h = self.net(cond)
        if self.t_update == "additive":
            shift = h
            return None, shift
        else:
            log_scale, shift = torch.chunk(h, 2, dim=-1)
            log_scale = self.scale_clip * torch.tanh(log_scale / self.scale_clip)
            return log_scale, shift

    def forward(self, x, return_logdet=False):
        s, u, t = self._split(x)

        if self.mode == "s_odd":
            s_new = s.clone()
            cond = torch.cat([s[:, self.even_idx], u, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            s_new[:, self.odd_idx] = s[:, self.odd_idx] * torch.exp(log_scale) + shift
            y = self._merge(s_new, u, t)
            logdet = log_scale.sum(dim=-1)

        elif self.mode == "s_even":
            s_new = s.clone()
            cond = torch.cat([s[:, self.odd_idx], u, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            s_new[:, self.even_idx] = s[:, self.even_idx] * torch.exp(log_scale) + shift
            y = self._merge(s_new, u, t)
            logdet = log_scale.sum(dim=-1)

        elif self.mode == "u_odd":
            u_new = u.clone()
            cond = torch.cat([u[:, self.even_idx], s, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            u_new[:, self.odd_idx] = u[:, self.odd_idx] * torch.exp(log_scale) + shift
            y = self._merge(s, u_new, t)
            logdet = log_scale.sum(dim=-1)

        elif self.mode == "u_even":
            u_new = u.clone()
            cond = torch.cat([u[:, self.odd_idx], s, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            u_new[:, self.even_idx] = u[:, self.even_idx] * torch.exp(log_scale) + shift
            y = self._merge(s, u_new, t)
            logdet = log_scale.sum(dim=-1)

        else:  # mode == "t"
            cond = torch.cat([s, u], dim=-1)
            log_scale, shift = self._t_params(cond)

            if self.t_update == "additive":
                t_new = t + shift
                logdet = t.new_zeros(t.shape[0])
            else:
                t_new = t * torch.exp(log_scale) + shift
                logdet = log_scale.sum(dim=-1)

            y = self._merge(s, u, t_new)

        if return_logdet:
            return y, logdet
        return y

    def inverse(self, y, return_logdet=False):
        s, u, t = self._split(y)

        if self.mode == "s_odd":
            s_old = s.clone()
            cond = torch.cat([s[:, self.even_idx], u, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            s_old[:, self.odd_idx] = (s[:, self.odd_idx] - shift) * torch.exp(-log_scale)
            x = self._merge(s_old, u, t)
            logdet = (-log_scale).sum(dim=-1)

        elif self.mode == "s_even":
            s_old = s.clone()
            cond = torch.cat([s[:, self.odd_idx], u, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            s_old[:, self.even_idx] = (s[:, self.even_idx] - shift) * torch.exp(-log_scale)
            x = self._merge(s_old, u, t)
            logdet = (-log_scale).sum(dim=-1)

        elif self.mode == "u_odd":
            u_old = u.clone()
            cond = torch.cat([u[:, self.even_idx], s, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            u_old[:, self.odd_idx] = (u[:, self.odd_idx] - shift) * torch.exp(-log_scale)
            x = self._merge(s, u_old, t)
            logdet = (-log_scale).sum(dim=-1)

        elif self.mode == "u_even":
            u_old = u.clone()
            cond = torch.cat([u[:, self.odd_idx], s, t], dim=-1)
            log_scale, shift = self._affine_params(cond)
            u_old[:, self.even_idx] = (u[:, self.even_idx] - shift) * torch.exp(-log_scale)
            x = self._merge(s, u_old, t)
            logdet = (-log_scale).sum(dim=-1)

        else:  # mode == "t"
            cond = torch.cat([s, u], dim=-1)
            log_scale, shift = self._t_params(cond)

            if self.t_update == "additive":
                t_old = t - shift
                logdet = t.new_zeros(t.shape[0])
            else:
                t_old = (t - shift) * torch.exp(-log_scale)
                logdet = (-log_scale).sum(dim=-1)

            x = self._merge(s, u, t_old)

        if return_logdet:
            return x, logdet
        return x


class HierarchicalSemanticFlowMap(nn.Module):
    """
    Stack cycles of:
        s_odd  | (s_even, u, t)
        s_even | (s_odd,  u, t)
        u_odd  | (u_even, s, t)
        u_even | (u_odd,  s, t)
        t      | (s, u)

    Output ordering is always [s | u | t].
    """

    def __init__(
        self,
        p=None,
        dim=None,
        K=4,
        hidden_units=128,
        num_hidden_layers=2,
        scale_clip=2.0,
        t_update="additive",
    ):
        super().__init__()

        if p is None:
            if dim is None:
                raise ValueError("Either p or dim must be provided.")
            if (dim - 1) % 2 != 0:
                raise ValueError("dim must equal 2*p + 1.")
            p = (dim - 1) // 2

        self.p = int(p)
        self.dim = 2 * self.p + 1

        layers = []
        for _ in range(K):
            layers.append(
                HierarchicalSemanticCoupling(
                    p=self.p,
                    mode="s_odd",
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                    t_update=t_update,
                )
            )
            layers.append(
                HierarchicalSemanticCoupling(
                    p=self.p,
                    mode="s_even",
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                    t_update=t_update,
                )
            )
            layers.append(
                HierarchicalSemanticCoupling(
                    p=self.p,
                    mode="u_odd",
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                    t_update=t_update,
                )
            )
            layers.append(
                HierarchicalSemanticCoupling(
                    p=self.p,
                    mode="u_even",
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                    t_update=t_update,
                )
            )
            layers.append(
                HierarchicalSemanticCoupling(
                    p=self.p,
                    mode="t",
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                    t_update=t_update,
                )
            )

        self.layers = nn.ModuleList(layers)

    def forward(self, x, return_logdet=False):
        z = x
        if return_logdet:
            total_logdet = x.new_zeros(x.shape[0])

        for layer in self.layers:
            if return_logdet:
                z, logdet = layer(z, return_logdet=True)
                total_logdet = total_logdet + logdet
            else:
                z = layer(z)

        if return_logdet:
            return z, total_logdet
        return z

    def inverse(self, z, return_logdet=False):
        x = z
        if return_logdet:
            total_logdet = z.new_zeros(z.shape[0])

        for layer in reversed(self.layers):
            if return_logdet:
                x, logdet = layer.inverse(x, return_logdet=True)
                total_logdet = total_logdet + logdet
            else:
                x = layer.inverse(x)

        if return_logdet:
            return x, total_logdet
        return x

def build_flow_vi(X, y, sigma2, tau, K_q=8, K_g=8,
                  hidden_units=64, num_hidden_layers=2):
    p = X.shape[1]
    dim = 2 * p + 1

    g_theta = HierarchicalSemanticFlowMap(
        p=p,
        K=K_g,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
        scale_clip=2.0,
        t_update="affine",   # 我建议先用 additive
    )

    generative_model = Relaxedsas(
        X=X, y=y, sigma2=sigma2, tau=tau, g_theta=g_theta
    )

    q0 = NBase(dim=dim, init_loc=0.0, init_log_scale=-2.5)

    posterior_flow = HierarchicalSemanticFlowMap(
        p=p,
        K=K_q,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
        scale_clip=2.0,
        t_update="affine",
    )

    return FlowVI(q0=q0, posterior_flow=posterior_flow, generative_model=generative_model,)



In [20]:
out_flow = simflow(seed=123, n=120, p=100, snr=2.5, true_prop=0.1, anneal_ratio=0.8,
    tau_end=0.01, K_q=8, K_g=8, hidden_units=64, num_hidden_layers=2,
    epochs=4000, num_samples=128, lr=5e-5, grad_clip=3.0, print_every=500,
)

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 6.834272766113282, 'sigma': 2.614244205523516, 'active_idx': array([14, 23, 24, 28, 29, 45, 69, 74, 84, 90], dtype=int64), 'snr': 2.5}
[epoch 0001] loss=888.872192 tau=1.000000 grad_norm=275.205383 n_selected=6 near_boundary=100
[epoch 0500] loss=256.983490 tau=0.487559 grad_norm=116.408455 n_selected=12 near_boundary=1
[epoch 1000] loss=247.307983 tau=0.237372 grad_norm=91.285019 n_selected=10 near_boundary=2
[epoch 1500] loss=246.512970 tau=0.115567 grad_norm=333.860687 n_selected=12 near_boundary=4
[epoch 2000] loss=246.118073 tau=0.056264 grad_norm=988.732849 n_selected=13 near_boundary=12
[epoch 2500] loss=245.976044 tau=0.027393 grad_norm=2678.228027 n_selected=14 near_boundary=41
[epoch 3000] loss=246.256241 tau=0.013336 grad_norm=8867.204102 n_selected=16 near_boundary=71
[epoch 3500] loss=246.123093 tau=0.010000 grad_norm=11030.882812 n_selected=22 near_boundary=68
[epoch 4000] loss=246.158539 tau=0.010

In [21]:
out_flow3 = simflow(seed=123, n=120, p=100, snr=2.5, true_prop=0.1, anneal_ratio=0.5,
    tau_end=0.01, K_q=16, K_g=16, hidden_units=64, num_hidden_layers=2,
    epochs=10000, num_samples=128, lr=5e-5, grad_clip=3.0, print_every=500,
)

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 6.834272766113282, 'sigma': 2.614244205523516, 'active_idx': array([14, 23, 24, 28, 29, 45, 69, 74, 84, 90], dtype=int64), 'snr': 2.5}
[epoch 0001] loss=892.026978 tau=1.000000 grad_norm=380.457733 n_selected=4 near_boundary=100
[epoch 0500] loss=250.882446 tau=0.631481 grad_norm=132.051620 n_selected=5 near_boundary=1
[epoch 1000] loss=246.335114 tau=0.398401 grad_norm=95.042015 n_selected=5 near_boundary=0
[epoch 1500] loss=245.873047 tau=0.251351 grad_norm=192.383331 n_selected=5 near_boundary=1
[epoch 2000] loss=245.607849 tau=0.158577 grad_norm=231.250977 n_selected=7 near_boundary=5
[epoch 2500] loss=245.867249 tau=0.100046 grad_norm=262.552734 n_selected=11 near_boundary=7
[epoch 3000] loss=245.352081 tau=0.063119 grad_norm=520.996521 n_selected=11 near_boundary=10
[epoch 3500] loss=245.570007 tau=0.039822 grad_norm=538.288940 n_selected=16 near_boundary=27
[epoch 4000] loss=245.389160 tau=0.025123 grad_n

In [22]:
out_flow4 = simflow(seed=123, n=120, p=100, snr=2.5, true_prop=0.1, anneal_ratio=1,
    tau_end=0.1, K_q=16, K_g=16, hidden_units=64, num_hidden_layers=2,
    epochs=6000, num_samples=128, lr=5e-5, grad_clip=3.0, print_every=500,
)

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 6.834272766113282, 'sigma': 2.614244205523516, 'active_idx': array([14, 23, 24, 28, 29, 45, 69, 74, 84, 90], dtype=int64), 'snr': 2.5}
[epoch 0001] loss=892.026978 tau=1.000000 grad_norm=380.457733 n_selected=4 near_boundary=100
[epoch 0500] loss=250.807617 tau=0.825695 grad_norm=105.434967 n_selected=5 near_boundary=0
[epoch 1000] loss=246.271561 tau=0.681510 grad_norm=89.335526 n_selected=4 near_boundary=0
[epoch 1500] loss=245.824432 tau=0.562503 grad_norm=98.325775 n_selected=4 near_boundary=0
[epoch 2000] loss=245.568909 tau=0.464278 grad_norm=97.909630 n_selected=5 near_boundary=0
[epoch 2500] loss=245.601593 tau=0.383204 grad_norm=84.669380 n_selected=6 near_boundary=1
[epoch 3000] loss=245.114517 tau=0.316288 grad_norm=127.564682 n_selected=6 near_boundary=0
[epoch 3500] loss=245.506073 tau=0.261057 grad_norm=90.241180 n_selected=6 near_boundary=0
[epoch 4000] loss=245.310974 tau=0.215471 grad_norm=113.0